In [ ]:
import os
import matplotlib
matplotlib.use("Agg")
import pandas as pd
import numpy as np
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import random
from tqdm.auto import tqdm
import getpass
import gc
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device = {str(device).title()}")

script_dir = os.path.dirname(os.path.abspath(os.getcwd()))

df_path = os.path.join(script_dir, "Datasets","House Prices.csv")
df = pd.read_csv(df_path)
df = df.dropna()

x = df.drop(columns="house_price").values
y = df["house_price"].values.reshape(-1,1)

x_scaler = StandardScaler()
y_scaler = StandardScaler()
x = x_scaler.fit_transform(x)
y = y_scaler.fit_transform(y)

x_train,x_test,y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=21)
num_features = x_train.shape[1]

x_train = torch.tensor(x_train, dtype=torch.float32).to(device)
x_test = torch.tensor(x_test, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test, dtype=torch.float32).to(device)

print(f"X Train = {x_train.shape}")
print(f"Y Train = {y_train.shape}")
print(f"X Test = {x_test.shape}")
print(f"Y Test = {y_test.shape}")

user_account = getpass.getuser()
predictions_bar_chart = os.path.join(script_dir, f"House Price Predictor Bar Chart.png")

class Regressor(nn.Module):
    def __init__(self, input_dim=num_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(in_features=input_dim, out_features=8),
            nn.LeakyReLU(0.1),
            nn.BatchNorm1d(8),

            nn.Linear(in_features=8, out_features=18),
            nn.LeakyReLU(0.8),
            nn.BatchNorm1d(18),

            nn.Linear(in_features=18, out_features=29),
            nn.LeakyReLU(),
            nn.BatchNorm1d(29),

            nn.Linear(in_features=29, out_features=47),
            nn.LeakyReLU(),
            nn.BatchNorm1d(47),
            nn.Dropout(p=0.7),

            nn.Linear(in_features=47, out_features=52),
            nn.LeakyReLU(),
            nn.BatchNorm1d(52),
            nn.Dropout(p=0.9),

            nn.Linear(in_features=52, out_features=67),
            nn.LeakyReLU(),
            nn.BatchNorm1d(67),
            nn.Dropout(p=0.4),

            nn.Linear(in_features=68, out_features=46),
            nn.LeakyReLU(),
            nn.BatchNorm1d(46),

            nn.Linear(in_features=46, out_features=21),
            nn.LeakyReLU(0.4),
            nn.BatchNorm1d(21),

            nn.Linear(in_features=21, out_features=1)
        )

    def forward(self, x:torch.Tensor) -> torch.Tensor:
        return self.network(x)

losses = []
def run_grid_search(epochs_counts, learning_rates, seed_val):
    best_state = None        
    loss_fn = nn.L1Loss()

    print(f"\n== Leaderboard ==")
    print(f"{'Learning Rate':<15} | {'Epochs':<15} | {'Val Loss':<15}")
    print(f"=" * 65)

    for lr in learning_rates:
        for epoch_count in epochs_counts:
            torch.manual_seed(int(seed_val))
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(int(seed_val))

            model = Regressor().to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=lr)

            for epoch in tqdm(range(epoch_count)):
                model.train()
                y_preds = model(x_train)
                loss = loss_fn(y_preds, y_train)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            model.eval()
            with torch.inference_mode():
                test_loss = loss_fn(model(x_test), y_test)

            print(f"{lr:<15} | {epoch_count:<15} | {loss:<15}")

            losses.append(test_loss)
            import copy
            best_state = copy.deepcopy(model.state_dict())

            del model, optimizer
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    print(f"\n● Best Loss = {min(losses):.3f}")
    print(f"● Worst Loss = {max(losses):.3f}")

    return best_state

epochs_list = [1340, 9800, 1500, 780]
learning_rates = [0.0001, 0.1, 0.001, 0.2]

seed_val = float(input("-- Enter a Value for a Seed : ").strip())
best_state = run_grid_search(epochs_list, learning_rates, seed_val=seed_val)
torch.save(best_state, f"House Price Predictor Parameters.pth")

fresh_model = Regressor().to(device)
fresh_model.load_state_dict(best_state)
fresh_model.eval()

session_inputs = []
session_predictions = []

while True:
    user_name = input("-- Enter your Name ( or type 'exit' to quit ) : ").strip()

    if user_name.lower() == "exit":
        print(f"\nExiting Interactive Loop ...")
        break

    area_sqft = float(input("-- Enter the preferred Area of your House in Square Feet : ").strip())
    bedrooms = float(input("-- Enter the preferred amount of Bedrooms in your House : ").strip())
    bathrooms = float(input("-- Enter the preferred amount of Bathrooms in your House : ").strip())
    age_years = float(input("-- Enter the preferred Age of your House in Years : ").strip())
    distance_city_km = float(input("-- Enter the preferred distance of your House from the City in Km : ").strip())

    raw_array = np.array([area_sqft, bedrooms, bathrooms, age_years, distance_city_km])
    scaled_array = x_scaler.transform(raw_array)
    scaled_tensor = torch.tensor(scaled_array, dtype=torch.float32).to(device)

    with torch.inference_mode():
        preds = fresh_model(scaled_tensor).cpu().numpy()

    untransformed = y_scaler.inverse_transform(preds).item()
    print(f"Predicted House Price for you, {user_name.title()} = {round(untransformed)} $")    

    session_inputs.append(f"{user_name} ( {area_sqft} Square Feet )")
    session_predictions.append(round(untransformed))

try:
    if session_predictions and session_inputs:
        plt.figure(figsize=(15,6))
        colors = ["red","blue","green","orange","purple"]
        bar_colors = [random.choice(colors) for _ in range(len(session_predictions))]
        plt.bar(session_inputs, session_predictions, colors=bar_colors, edgecolor="black")
        plt.title("Predictions Bar Chart = House Price Predictor")
        plt.xlabel("People | Preferred House Size")
        plt.ylabel("Predicted House Prices")
        plt.grid()
        plt.legend(prop={"size":14,"weight":"bold"})
        plt.savefig(predictions_bar_chart)
        plt.close()
        print(f"Saved the Graph")
    else:
        print(f"Failed to Save Graph = No Inputs or Predictions")
except BaseException as e:
    print(f"Failed to Save Graph = {e}") 

one_drive = input("-- Do you use One Drive or Not ( Yes / No ) : ").strip().title()
try:
    if one_drive == "Yes":
        desktop_path = f"C:/Users/{user_account}/OneDrive/Desktop/House Price Predictor Details.txt"      
    elif one_drive == "No":
        desktop_path =  f"C:/Users/{user_account}/Desktop/House Price Predictor Details.txt" 
except BaseException as e:
    print(f"Please enter Valid Input. Error has occured = {e}")

try:
    with open(desktop_path, "w", encoding="utf-8") as file:
        file.write(f"== House Price Predictor Details ==\n")
        file.write(f"\n● Parameter Weights = {fresh_model.network[0].weight[0][0].item():.3f}\n")
        file.write(f"● Parameter Bias = {fresh_model.network[0].bias[0].item():.3f}")
        file.write(f"● Worst Loss = {min(losses):.2f}\n")
        file.write(f"● Worst Loss = {max(losses):.3f}\n")
    print(f"Sucessfully Saved your Model Details")
except BaseException as e:
    print(f"Failed to Save Network Details = {e}")        

c:\Users\there\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using Device = Cuda
X = (15000, 5)
Y = (15000, 1)
X Train = torch.Size([12000, 5])
Y Train = torch.Size([12000, 1])
X Test = torch.Size([3000, 5])
Y Test = torch.Size([3000, 1])

== Leaderboard ==
Learning Rate   | Epochs          | Val Loss       


  0%|          | 0/1340 [00:00<?, ?it/s]


RuntimeError: mat1 and mat2 shapes cannot be multiplied (12000x67 and 68x46)